# 矢量拟合以下是对矢量拟合的简要介绍。该概念及其算法由 Bjørn Gustavsen 和 Adam Semlyen 于 1999 年提出 [[1](#link_ref1)]。有关更多信息，请参阅矢量拟合网站 [[2](#link_ref2)]。矢量拟合的主要应用是在电路仿真器中对有源或无源器件的原始采样频率响应进行建模。此外，请参阅 [API 文档](../api/vectorFitting.html) 和 [应用示例](../examples/index.html#vector-fitting)，以获取有关 scikit-rf 中实现的更多信息。

## 数学描述矢量拟合法旨在将一组有理模型函数拟合到一组采样频率响应 $\mathbf{\underline{H}}_\mathrm{sampled}$，例如来自 [S](https://en.wikipedia.org/wiki/Scattering_parameters)、[Y](https://en.wikipedia.org/wiki/Admittance_parameters) 或 [Z](https://en.wikipedia.org/wiki/Impedance_parameters) 矩阵的数据。模型函数 $\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 定义在拉普拉斯域中，其中 $\mathrm{\underline{s}} = \sigma + \mathrm{j} \omega$：\begin{equation}\mathbf{\underline{H}}(\mathrm{\underline{s}}) = \mathbf{d} + \mathrm{\underline{s}} \mathbf{e} + \sum_{k=1}^K \frac{\underline{\mathbf{c}}_{k}}{\mathrm{\underline{s}}-\underline{p}_k}\end{equation}为了实现所需的拟合效果，该模型函数应在每个频率采样点 $\omega_n$ 处匹配给定的频率响应：\begin{equation}\mathbf{\underline{H}}(\mathrm{\underline{s}} = \mathrm{j} \omega_n) \overset{!}{=} \mathbf{\underline{H}}_\mathrm{sampled}(\omega_n)\end{equation}通常，$\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 是一个向量，包含模型中各个复数频率响应 $\underline{H}_1(\mathrm{\underline{s}})$, $\underline{H}_2(\mathrm{\underline{s}})$, ..., $\underline{H}_M(\mathrm{\underline{s}})$。$\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 中的所有元素共享一组公共的复数极点 $\underline{p}_k$，但具有各自的复数残差 $\mathbf{\underline{c}}_k$、实数常数 $\mathbf{d}$ 和实数比例系数 $\mathbf{e}$，因此它们也是向量：\begin{equation}\mathbf{\underline{p}} = \begin{pmatrix} \underline{p}_1 & \underline{p}_2 & \underline{p}_3 & \cdots & \underline{p}_K \end{pmatrix}\end{equation}\begin{equation}\mathbf{\underline{c}} = \begin{pmatrix} \underline{c}_{1,1} & \underline{c}_{2,1} & \underline{c}_{3,1} & \cdots & \underline{c}_{K,1} \\\underline{c}_{1,2} & \underline{c}_{2,2} & \underline{c}_{3,2} & \cdots & \underline{c}_{K,2} \\\vdots\\\underline{c}_{1,M} & \underline{c}_{2,M} & \underline{c}_{3,M} & \cdots & \underline{c}_{K,M} \\\end{pmatrix}\end{equation}\begin{equation}\mathbf{d} = \begin{pmatrix} d_1 \\ d_2 \\ \vdots \\ d_M \end{pmatrix}\end{equation}\begin{equation}\mathbf{e} = \begin{pmatrix} e_1 \\ e_2 \\ \vdots \\ e_M \end{pmatrix}\end{equation}为了使 $\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 能够很好地拟合 $\mathbf{\underline{H}}_\mathrm{sampled}$，所需的极点数 $K$ 取决于响应的形状。例如，目标可能是将有理模型函数拟合到 2 端口 ($M = 4$) 的 S 矩阵，该矩阵在 $N$ 个频率 $\omega_n$ 处采样：\begin{equation}\begin{pmatrix} \underline{S}_{11}(\omega_1) \\\underline{S}_{12}(\omega_1) \\\underline{S}_{21}(\omega_1) \\\underline{S}_{22}(\omega_1) \\\vdots \\\underline{S}_{11}(\omega_\mathrm{N}) \\\underline{S}_{12}(\omega_\mathrm{N}) \\\underline{S}_{21}(\omega_\mathrm{N}) \\\underline{S}_{22}(\omega_\mathrm{N})\end{pmatrix}\overset{!}{=}\begin{pmatrix}d_{11} + \mathrm{j} \omega_1 e_{11} + \sum_{k=1}^K \frac{\underline{c}_{k,11}}{\mathrm{j} \omega_1 - \underline{p}_k}\\d_{12} + \mathrm{j} \omega_1 e_{12} + \sum_{k=1}^K \frac{\underline{c}_{k,12}}{\mathrm{j} \omega_1 - \underline{p}_k}\\d_{21} + \mathrm{j} \omega_1 e_{21} + \sum_{k=1}^K \frac{\underline{c}_{k,21}}{\mathrm{j} \omega_1 - \underline{p}_k}\\d_{22} + \mathrm{j} \omega_1 e_{22} + \sum_{k=1}^K \frac{\underline{c}_{k,22}}{\mathrm{j} \omega_1 - \underline{p}_k}\\\vdots\\d_{11} + \mathrm{j} \omega_\mathrm{N} e_{11} + \sum_{k=1}^K \frac{\underline{c}_{k,11}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}\\d_{12} + \mathrm{j} \omega_\mathrm{N} e_{12} + \sum_{k=1}^K \frac{\underline{c}_{k,12}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}\\d_{21} + \mathrm{j} \omega_\mathrm{N} e_{21} + \sum_{k=1}^K \frac{\underline{c}_{k,21}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}\\d_{22} + \mathrm{j} \omega_\mathrm{N} e_{22} + \sum_{k=1}^K \frac{\underline{c}_{k,22}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}\end{pmatrix}\end{equation}在矢量拟合过程中，模型参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$ 将以迭代方式进行优化，直到获得良好的拟合效果。

## 等效电路对采样频率响应进行矢量拟合的好处在于，它可以很容易地用等效电路来表示有理基函数。 详细推导可以在 [[3](#link_ref3)] 和 [[4](#link_ref4)] 中找到。### 常数项和比例项常数项和比例项 $\mathbf{d} + \mathrm{\underline{s}} \mathbf{e}$ 可以通过等效阻抗 $\underline{Z}_\mathrm{RL}(\mathrm{\underline{s}})$ 或等效导纳 $\underline{Y}_\mathrm{RC}(\mathrm{\underline{s}})$ 来表示，这些阻抗或导纳由串联 RL 电路或并联 RC 电路构成。<center><img src='./figures/circuit_series_RL.svg' width='170px'></center><center><img src='./figures/circuit_parallel_RC.svg' width='150px'></center>常数项和比例项的目标响应：\begin{equation}\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = d_i + \mathrm{\underline{s}} e_i\end{equation}#### 串联 RL 电路的阻抗：\begin{equation}\underline{Z}_\mathrm{RL}(\mathrm{\underline{s}}) = R + \mathrm{\underline{s}} L\end{equation}如果 $R = d_i$ 且 $L = e_i$，则此阻抗与目标响应匹配。#### 并联 RC 电路的导纳：\begin{equation}\underline{Y}_\mathrm{RC}(\mathrm{\underline{s}}) = \frac{1}{R} + \mathrm{\underline{s}} C\end{equation}如果 $R = \frac{1}{d_i}$ 且 $C = e_i$，则此导纳与目标响应匹配。### 实极点和残差拟合中的每个项 $\frac{c_{k,i}}{\mathrm{\underline{s}} - p_{k,i}}$，其中包含一个实极点 $p_{k,i}$ 和一个实残差 $c_{k,i}$，可以通过等效阻抗 $\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}})$ 或等效导纳 $\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}})$ 来表示，这些阻抗或导纳由并联 RC 电路或串联 RL 电路构成。实极点-残差项的目标响应：\begin{equation}\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = \frac{c_{k,i}}{\mathrm{\underline{s}} - p_{k,i}}\end{equation}#### 并联 RC 电路的阻抗：并联 RC 电路与常数项和比例项中的相同，但这一次使用其阻抗 $\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}})$，而不是其导纳：\begin{equation}\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}}) = \frac{\frac{1}{C}}{\mathrm{\underline{s}} + \frac{1}{RC}}\end{equation}如果 $C = \frac{1}{c_{k,i}}$ 且 $R = \frac{c_{k,i}}{-p_{k,i}}$，则此阻抗与目标响应匹配。#### 串联 RL 电路的导纳：串联 RL 电路与常数项和比例项中的相同，但这一次使用其导纳 $\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}})$，而不是其阻抗：\begin{equation}\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}}) = \frac{\frac{1}{L}}{\mathrm{\underline{s}} + \frac{R}{L}}\end{equation}如果 $L = \frac{1}{c_{k,i}}$ 且 $R = \frac{-p_{k,i}}{c_{k,i}}$，则此导纳与目标响应匹配。### 共轭复极点和残差在矢量拟合中，复极点 $\underline{p}_k = p'_k + \mathrm{j} p''_k$ 和残差 $\underline{c}_k = c'_k + \mathrm{j} c''_k$ 总是以共轭复数对 $(\underline{p}_k, \underline{p}_k^*)$ 和 $(\underline{c}_k, \underline{c}_k^*)$ 出现。 因此，此类共轭复数对的等效电路的目标响应是：\begin{equation}\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = \frac{\underline{c}_k}{\mathrm{\underline{s}} - \underline{p}_k} + \frac{\underline{c}^*_k}{\mathrm{\underline{s}} - \underline{p}^*_k} = \frac{2 c'_k \mathrm{\underline{s}} - 2 (c'_k p'_k + c''_k p''_k)}{\mathrm{\underline{s}}^2 - 2 p'_k \mathrm{\underline{s}} + |\underline{p}_k|^2}\end{equation}可以使用无源或有源元件构建等效电路，并可以对其进行调整以匹配此目标频率响应。 在 [[3](#link_ref3)] 中介绍了四种此类电路并进行了分析，其中两种在此处介绍。#### 并联 RLC 电路的阻抗一个并联 RLC 电路，其电感器串联一个电阻，可以提供一个阻抗 $\underline{Z}_\mathrm{RCL}(\mathrm{\underline{s}})$，该阻抗可以与目标响应匹配。<center><img src='./figures/circuit_cc-pole_rlc-passive.svg' width='200px'></center>\begin{equation}\underline{Z}_\mathrm{RCL}(\mathrm{\underline{s}}) = \frac{\frac{1}{C} \mathrm{\underline{s}} + \frac{R_1}{LC}}{\mathrm{\underline{s}}^2 + (\frac{1}{R_2 C}) \mathrm{\underline{s}} + \frac{1}{LC} (1 + \frac{R_1}{R_2})}\end{equation}如果 $C = \frac{1}{2 c'_k}$、$L = \frac{2 c'_k}{(p''_k)^2 + (\frac{c''_k p''_k}{c'_k})^2}$、$R_1 = \frac{-2(c'_k p'_k + c''_k p''_k)}{(p''_k)^2 + (\frac{c''_k p''_k}{c'_k})^2}$，且 $R_2 = \frac{(2 c'_k)^2}{-2(c'_kp'_k-c''_kp''_k)}$，则此阻抗与目标响应匹配。在某些情况下，计算出的电阻可能为负值，这意味着所需的无源电路必须提供增益，这通常是不可能的。 如果建模的网络是有源网络，则这种情况并不少见，但对于纯无源网络也可能发生。 一些电路仿真器允许使用受控源来实现负电阻。 为了完全控制行为，合成的等效电路已经包含与电阻并联的电压控制电流源，以实现负有效电阻。<center><img src='./figures/circuit_cc-pole_rlc-active.svg' width='200px'></center>与电阻 $R_i$ 并联的电流源的有效电阻 $R_{i,eff}$，该电流源由电阻上的电压 $V_i$ 控制，等于 $-R_i$，如果受控电流源的跨导为 $g_{T,i} = \frac{-2}{R_i}$。这样，两个电阻 $R_1$ 和 $R_2$ 都可以通过正值和负值来实现。 在正电阻的情况下，我们只需将 $g_{T,i} = 0$ 即可。#### 串联 RLC 电路的导纳一个串联 RLC 电路与并联的电流源结合使用，该电流源由电容器上的电压控制，可以提供一个导纳 $\underline{Y}_\mathrm{RCL,I}(\mathrm{\underline{s}})$，该导纳可以与目标响应匹配。<center><img src='./figures/circuit_series_RCL_parallel_current.svg' width='200px'></center>\begin{equation}\underline{Y}_\mathrm{RCL,I}(\mathrm{\underline{s}}) = \frac{1/L \mathrm{\underline{s}} + b}{\mathrm{\underline{s}}^2 + R/L \mathrm{\underline{s}} + 1 / (LC)}\end{equation}如果 $L = \frac{1}{2 c'_k}$、$R = \frac{-p'_k}{ c'_k}$、$C = \frac{2 c'_k}{|\underline{p}_k|^2}$ 且 $b = -2 (c'_k p'_k + c''_k p''_k)$，且 $g_\mathrm{m} = bLC = \frac{b}{|\underline{p}_k|^2}$，则此导纳与目标响应匹配。

## 矢量拟合 N 端口的等效电路### 情况 1：矢量拟合的散射参数具有矢量拟合散射矩阵的 N 端口的等效电路由一个接口网络和每个端口的传输网络组成。对于接口网络的实现（有时也称为端口网络），可以使用一个电流源 ($I_\mathrm{src,i}$) 和一个并联电阻 $R_i$（诺顿等效），或者使用一个电压源 ($V_\mathrm{src,i}$) 和一个串联电阻 $R_i$（戴维南等效）。求解诺顿等效接口网络的基尔霍夫电流定律，得到 $V_i = R_i (I_i + I_{src,i})$。求解戴维南等效接口网络的基尔霍夫电压定律，得到 $V_i = R_i I_i + V_{src,i}$。将这两个方程与端口 $i$ 处入射波 ($a_i$) 和反射波 ($b_i$) 的定义进行比较，可以发现，源项 $I_\mathrm{src,i}$ 和 $V_\mathrm{src,i}$ 与反射波 $b_i$ 成正比，其中 $R_i$ 是端口 $i$ 的参考阻抗：\begin{equation}b_i = \frac{1}{2 \sqrt{R_i}} (V_i - R_i I_i) = \frac{\sqrt{R_i}}{2} I_{src,i} = \frac{1}{2 \sqrt{R_i}} V_{src,i}\end{equation}入射波 $a_i$ 的计算方式是，根据端口电压 $V_i$ 和端口电流 $I_i$ 进行计算，与接口网络的类型无关：\begin{equation}a_i = \frac{1}{2\sqrt{R_i}} (V_i + R_i I_i)\end{equation}#### 使用导纳表示散射响应下图显示了这样一个端口 $i$ 的接口网络和传输网络的结构，其中包含外部端口电压 $V_i$ 和端口电流 $I_i$。矢量拟合的各个频率响应 $\underline{S}_{i,j}$ 通过等效导纳 $\underline{Y}_{\mathrm{S},i,j}$ 来复现，该导纳基于拟合参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$，如上所述。<center><img src='./figures/circuit_equivalent_s_admittance.svg' width='500px'></center>这种等效电路拓扑结构有一些可优化的空间。例如，用于驱动并联导纳堆叠的入射波的受控源，可以实现为差分形式，并将产生的电流加总在一个节点中，然后将其传输回各自的端口，以得到反射波。下图显示了这种方案：<center><img src='./figures/circuit_equivalent_s_admittance_differential.png' width='500px'></center>#### 使用阻抗表示散射响应下图显示了这样一个端口 $i$ 的接口网络和传输网络的结构，其中包含外部端口电压 $V_i$ 和端口电流 $I_i$。矢量拟合的各个频率响应 $\underline{S}_{i,j}$ 通过等效阻抗 $\underline{Z}_{\mathrm{S},i,j}$ 来复现，该导纳基于拟合参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$，如上所述。阻抗由受控电流源驱动，这些电流源产生与入射波 $a_i$ 成正比的电流，其中 $g_{a,i} = \frac{1}{2 \sqrt{R_i}}$ 和 $r_{a,i} = \frac{\sqrt{R_i}}{2}$。分别表示由单个极点-残差对建模的总散射响应一部分的各个阻抗 $Z_{j,i,k}$ 上的电压 $V_{j,i,k}$，被传输到端口接口处的相应电压控制电流源，其增益为 $g_{b,j} = \frac{\sqrt{R_i}}{2}$，如上所述。<center><img src='./figures/circuit_equivalent_s_impedance.svg' width='600px'></center>#### 直接实现状态空间方程从状态空间模型直接综合等效电路是相当直接的，因为电路拓扑结构是从 [状态空间方程](https://en.wikipedia.org/wiki/State-space_representation) 推导出来的：\begin{equation}\mathbf{\dot{x}}(t) = \mathbf{A} \mathbf{x}(t) + \mathbf{B} \mathbf{u}(t)\end{equation}\begin{equation}\mathbf{y}(t) = \mathbf{C} \mathbf{x}(t) + \mathbf{D} \mathbf{u}(t) + \mathbf{E} \mathbf{\dot{u}}(t)\end{equation}对于散射参数，输入向量 $\mathbf{u}$ 表示入射波 $\mathbf{a}$，输出向量 $\mathbf{y}$ 表示反射波 $\mathbf{b}$。 无论参数表示如何，状态向量 $\mathbf{x}$ 都是一个内部变量，用于定义网络的动态特性。在等效电路中，电路元件的值由状态空间矩阵 A、B、C、D（以及 E）中的系数给出。 由于在矢量拟合的极点/残差模型中，这些矩阵的稀疏性，非零电路元件的数量远低于完全填充的情况，从而产生一个紧凑的电路网表。如图所示，每个端口都有一个接口网络，将各个输入和状态及其各自的增益传输到输出。状态网络取决于极点的类型：一个实极点由一个状态表示，一个复共轭极点对由两个状态表示。 在可选的比例项 E 的情况下，每个端口都需要另一个微分网络。<center><img src='./figures/circuit_equivalent_s_statespace.svg' width='600px'></center>各个受控源的增益如下，主要来自散射参数的定义，该定义是端口电压和电流的函数：- $g_{D,i,j} = \frac{2}{\sqrt{R_{0,i}}} d_{i,j} \frac{1}{2\sqrt{R_{0,j}}}$- $f_{D,i,j} = \frac{2}{\sqrt{R_{0,i}}} d_{i,j} \frac{\sqrt{R_{0,j}}}{2}$- $g_{E,i,j} = \frac{2}{\sqrt{R_{0,i}}} e_{i,j}$- $g_{C,i,j,k} = \frac{2}{\sqrt{R_{0,i}}} c_{i,j,k}$- $g_{a,j} = \frac{1}{2\sqrt{R_{0,j}}}$- $f_{a,j} = \frac{\sqrt{R_{0,j}}}{2}$有关状态空间电路综合的更多详细信息，请参阅 S. Grivet-Talocia 和 B. Gustavsen 的《被动宏建模》[[4](#link_ref4)]。#### 在 scikit-rf 中的实现scikit-rf 中的实现过去已经改变了几次，尝试使用等效导纳、等效阻抗和直接状态空间电路综合，如上所示。 目前，scikit-rf 使用状态空间方程的直接实现。 通过比较 ngspice、Xyce 和 LTspice 中的仿真运行时间，可以发现不同类型等效电路网表之间存在很大差异。 尽管很难查明性能差异的确切原因，但似乎受控源的类型对基于修正节点分析 (MNA) 的任何仿真都有很大的影响。 电压控制电流源通常比受控电压源表现更好。 令人惊讶的是，对于大多数 MNA 电路求解器来说，受控源的数量并不总是至关重要的，只要控制信号已经存在于线性系统的解向量中。 因此，合成的等效电路网表的文件大小并不能代表仿真器中的性能。### 情况 2：矢量拟合的 Y 参数尚未实现。 请参阅 [Y 参数](https://en.wikipedia.org/wiki/Admittance_parameters)，以了解通用的等效电路。### 情况 3：矢量拟合的 Z 参数尚未实现。 请参阅 [Z 参数](https://en.wikipedia.org/wiki/Impedance_parameters)，以了解通用的等效电路。

## References<div id='link_ref1'></div>[1] B. Gustavsen and A. Semlyen, "Rational approximation of frequency domain responses by vector fitting," in IEEE Transactions on Power Delivery, vol. 14, no. 3, pp. 1052-1061, July 1999, DOI: [10.1109/61.772353](https://doi.org/10.1109/61.772353).<div id='link_ref2'></div>[2] https://www.sintef.no/projectweb/vectorfitting/<div id='link_ref3'></div>[3] G. Antonini, "SPICE equivalent circuits of frequency-domain responses," in IEEE Transactions on Electromagnetic Compatibility, vol. 45, no. 3, pp. 502-512, Aug. 2003, DOI: [10.1109/TEMC.2003.815528](https://doi.org/10.1109/TEMC.2003.815528).<div id='link_ref4'></div>[4] "Passive Macromodeling" by S. Grivet-Talocia and B. Gustavsen, Wiley, 2016, DOI: [10.1002/9781119140931](https://doi.org/10.1002/9781119140931).